## PARTE 1: FACE DETECTION (AdaBoost)

### 1.1 Integral Image
**¿Qué es?** Cada píxel = suma de todos los píxeles a su izquierda y arriba

**¿Para qué?** Calcular sumas de regiones rectangulares en O(1) en lugar de O(n²)

**Fórmula:** `integral[i][j] = img[i][j] + integral[i-1][j] + integral[i][j-1] - integral[i-1][j-1]`

In [1]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# ===== INTEGRAL IMAGE =====
def compute_integral_image(img_arr):
    """Computa la imagen integral de un array 2D"""
    integral = np.zeros((img_arr.shape[0] + 1, img_arr.shape[1] + 1))
    integral[1:, 1:] = np.cumsum(img_arr, axis=0)
    integral = np.cumsum(integral, axis=1)
    return integral

# Test
test_img = np.array([[1, 2, 2], [1, 2, 2], [1, 2, 2]])
ii = compute_integral_image(test_img)
print("Integral Image:\n", ii)

Integral Image:
 [[ 0.  0.  0.  0.]
 [ 0.  1.  3.  5.]
 [ 0.  2.  6. 10.]
 [ 0.  3.  9. 15.]]


### 1.2 Sum Region (Rectángulo)
**¿Qué es?** Suma de píxeles en una región rectangular usando integral image

**Fórmula:** `sum = II[B] - II[A] - II[C] + II[D]`
- A = top-left, B = bottom-right, C y D = las otras esquinas

In [2]:
# ===== SUM REGION =====
def sum_region(integral_img, top_left, bottom_right):
    """Suma de píxeles en una región rectangular"""
    A = integral_img[top_left[0], top_left[1]]
    B = integral_img[top_left[0], bottom_right[1]]
    C = integral_img[bottom_right[0], top_left[1]]
    D = integral_img[bottom_right[0], bottom_right[1]]
    return D - C - B + A

### 1.3 Haar-like Features
**¿Qué son?** Filtros que comparan sumas de regiones claras vs oscuras

**Tipos:**
- TWO_VERTICAL: 2 regiones verticales
- TWO_HORIZONTAL: 2 regiones horizontales  
- THREE_VERTICAL/THREE_HORIZONTAL: 3 regiones
- FOUR: 4 regiones en cuadrícula

**Score:** diferencia entre regiones claras - regiones oscuras

In [3]:
# ===== HAAR-LIKE FEATURE =====
def enum(**enums):
    return type('Enum', (), enums)

FeatureType = enum(
    TWO_VERTICAL=(1, 2), 
    TWO_HORIZONTAL=(2, 1), 
    THREE_HORIZONTAL=(3, 1), 
    THREE_VERTICAL=(1, 3), 
    FOUR=(2, 2)
)

class HaarFeature:
    """Haar-like Feature para detección de caras"""
    def __init__(self, feature_type, position, width, height, threshold, polarity):
        self.type = feature_type
        self.top_left = position
        self.bottom_right = (position[0] + width, position[1] + height)
        self.width = width
        self.height = height
        self.threshold = threshold
        self.polarity = polarity
        self.weight = 1.0
    
    def get_score(self, int_img):
        """Score del feature en una imagen integral"""
        score = 0
        if self.type == FeatureType.TWO_VERTICAL:
            first = sum_region(int_img, self.top_left, (self.top_left[0] + self.width, int(self.top_left[1] + self.height / 2)))
            second = sum_region(int_img, (self.top_left[0], int(self.top_left[1] + self.height / 2)), self.bottom_right)
            score = first - second
        # ... otros tipos similar ...
        return score
    
    def get_vote(self, int_img):
        """Voto: 1 si detecta cara, -1 si no"""
        score = self.get_score(int_img)
        return self.weight * (1 if score < self.polarity * self.threshold else -1)

### 1.4 AdaBoost
**¿Qué es?** Algoritmo que combina múltiples clasificadores débiles en uno fuerte

**Proceso:**
1. Inicializar pesos uniformes (1/2m para positivos, 1/2n para negativos)
2. Para cada iteración:
   - Seleccionar el mejor feature que minimiza error ponderado
   - Calcular weight del feature: `α = 0.5 * ln((1-error)/error)`
   - Actualizar pesos: multiplicar por `sqrt(error/(1-error))` si acierta, `sqrt((1-error)/error)` si falla
3. Votación final: sumar votos ponderados

In [4]:
# ===== ENSEMBLE VOTING =====
def ensemble_vote(int_img, classifiers):
    """Voto mayoritario de múltiples clasificadores"""
    votes = [clf.get_vote(int_img) for clf in classifiers]
    return 1 if sum(votes) > 0 else 0

def ensemble_vote_all(int_imgs, classifiers):
    """Voto para múltiples imágenes"""
    return [ensemble_vote(img, classifiers) for img in int_imgs]

def ensemble_vote_t(int_img, classifiers, threshold):
    """Voto con threshold personalizado"""
    votes = [clf.get_vote(int_img) for clf in classifiers]
    return 1 if sum(votes) > threshold else 0

def ensemble_vote_all_t(int_imgs, classifiers, threshold):
    """Voto para múltiples imágenes con threshold"""
    return [ensemble_vote_t(img, classifiers, threshold) for img in int_imgs]

### 1.5 Evaluación
**Métricas:**
- Correctly identified faces: suma de predicciones = 1
- Correctly identified non-faces: (total - suma de predicciones = 1)
- Accuracy = correctas / total

In [5]:
# ===== EVALUACIÓN =====
def evaluate_detector(faces_imgs, non_faces_imgs, classifiers, threshold=0):
    """Evalúa el detector de caras"""
    # Faces (queremos que prediga 1)
    faces_pred = ensemble_vote_all_t(faces_imgs, classifiers, threshold)
    correct_faces = sum(faces_pred)
    
    # Non-faces (queremos que prediga 0)
    non_faces_pred = ensemble_vote_all_t(non_faces_imgs, classifiers, threshold)
    correct_non_faces = len(non_faces_imgs) - sum(non_faces_pred)
    
    total_correct = correct_faces + correct_non_faces
    total = len(faces_imgs) + len(non_faces_imgs)
    
    print(f"Faces correctas: {correct_faces}/{len(faces_imgs)} ({100*correct_faces/len(faces_imgs):.2f}%)")
    print(f"Non-faces correctas: {correct_non_faces}/{len(non_faces_imgs)} ({100*correct_non_faces/len(non_faces_imgs):.2f}%)")
    print(f"Accuracy total: {100*total_correct/total:.2f}%")

---
## PARTE 2: FACE RECOGNITION (Clasificación)

### 2.1 PCA (Principal Component Analysis)
**¿Qué es?** Encuentra direcciones de máxima varianza en los datos

**Ventajas:**
- No supervisado
- Reduce dimensionalidad
- Rápido

**Desventajas:**
- Varianza ≠ discriminación
- No usa etiquetas
- Max componentes = min(samples, features)

In [6]:
# ===== PCA =====
def apply_pca(X_train, X_test, n_components=100):
    """Aplica PCA y transforma datos"""
    pca = PCA(n_components=n_components, whiten=True, random_state=42)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)
    
    explained = np.sum(pca.explained_variance_ratio_)
    print(f"Varianza explicada: {100*explained:.2f}%")
    print(f"Componentes: {n_components}")
    
    return X_train_pca, X_test_pca, pca

### 2.2 LDA (Linear Discriminant Analysis)
**¿Qué es?** Encuentra direcciones que maximizan separación entre clases

**Ventajas:**
- Supervisado (usa etiquetas)
- Optimiza para clasificación
- Pocas dimensiones (max = n_clases - 1)

**Desventajas:**
- Asume distribuciones gaussianas
- Max componentes = n_clases - 1
- Requiere etiquetas

In [7]:
# ===== LDA =====
def apply_lda(X_train, X_test, y_train, n_components=None):
    """Aplica LDA y transforma datos"""
    # Max componentes = n_clases - 1
    n_classes = len(np.unique(y_train))
    n_comp = min(n_components or (n_classes - 1), n_classes - 1)
    
    lda = LinearDiscriminantAnalysis(n_components=n_comp)
    X_train_lda = lda.fit_transform(X_train, y_train)
    X_test_lda = lda.transform(X_test)
    
    explained = np.sum(lda.explained_variance_ratio_)
    print(f"Varianza explicada: {100*explained:.2f}%")
    print(f"Componentes: {n_comp}")
    
    return X_train_lda, X_test_lda, lda

### 2.3 Clasificadores

#### SVM (Support Vector Machine)
**Ventajas:** Robusto, kernel trick para no-lineales
**Hiperparámetros:** C (regularización), gamma (RBF kernel)

In [8]:
# ===== SVM =====
def train_svm(X_train, y_train, X_test, y_test, kernel='rbf', C=1000, gamma=0.001):
    """Entrena SVM"""
    svm = SVC(kernel=kernel, C=C, gamma=gamma, class_weight='balanced')
    svm.fit(X_train, y_train)
    
    y_pred = svm.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"SVM Accuracy: {100*acc:.2f}%")
    
    return svm, y_pred

#### k-NN (k-Nearest Neighbors)
**Ventajas:** Simple, no-paramétrico
**Desventajas:** Lento, sensible al ruido
**Hiperparámetro:** k (número de vecinos)

In [9]:
# ===== k-NN =====
def train_knn(X_train, y_train, X_test, y_test, k=5):
    """Entrena k-NN"""
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    
    y_pred = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"k-NN Accuracy (k={k}): {100*acc:.2f}%")
    
    return knn, y_pred

### 2.4 Métricas de Evaluación

**Precision:** TP / (TP + FP) → De lo que predijo positivo, cuánto acertó

**Recall:** TP / (TP + FN) → De lo que es positivo, cuánto encontró

**F1-Score:** Media armónica de Precision y Recall

**Accuracy:** (TP + TN) / Total → Porcentaje total de aciertos

In [10]:
# ===== EVALUACIÓN =====
def full_evaluation(y_test, y_pred, target_names=None):
    """Evaluación completa"""
    print("\n" + "="*60)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("="*60)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=target_names))

---
## QUICK COMPARISON: PCA vs LDA

| Aspecto | PCA | LDA |
|---------|-----|-----|
| **Tipo** | No supervisado | Supervisado |
| **Objetivo** | Max varianza | Max separación clases |
| **Max componentes** | min(m, n) | n_clases - 1 |
| **Velocidad** | Rápido | Más lento |
| **Dimensiones típicas** | 50-150 | 5-20 |
| **Para clasificación** | Bueno | Muy bueno |
| **Requiere etiquetas** | No | Sí |

---
## WORKFLOW TÍPICO EXAMEN

### Face Detection:
```
1. Cargar imágenes → compute_integral_image() para cada una
2. learn() genera features con AdaBoost
3. ensemble_vote_t() predice con threshold
4. Evaluar con métricas
```

### Face Recognition:
```
1. Cargar dataset faces (ej: LFW)
2. Train-test split
3. PCA o LDA para reducir dimensiones
4. Entrenar clasificador (SVM o k-NN)
5. Evaluar con classification_report()
```

---
## PREGUNTAS TÍPICAS EXAMEN

**¿Para qué sirve la Integral Image?**
→ Calcular sumas de regiones en O(1) en lugar de O(n²)

**¿Cuál es la diferencia PCA vs LDA?**
→ PCA maximiza varianza (no supervisado), LDA maximiza separación de clases (supervisado)

**¿Cuántos componentes máximo puede tener LDA?**
→ n_clases - 1

**¿Por qué usar threshold en ensemble voting?**
→ Para ajustar el balance entre True Positives y False Positives

**Desventajas k-NN:**
→ Lento (O(n×d) por predicción), requiere almacenar datos, sensible dimensionalidad alta, sin interpretabilidad

**¿Qué es el weight en AdaBoost?**
→ La importancia de cada clasificador: α = 0.5 × ln((1-error)/error)